# Notebook 121: バニラRNN ― 状態を持つネットワーク

## Vanilla RNN: Network with State

---

### このノートブックの位置づけ

**Unit 8「シーケンスモデリング」** の第2回として、時系列データを扱うための基本アーキテクチャである **Vanilla RNN（バニラRNN）** をNumPyのみで実装します。

### 学習目標

- [ ] RNNの **状態方程式** を理解し、数式と対応づけて説明できる
- [ ] **重み共有** の仕組みと利点を理解する
- [ ] **展開計算グラフ** を描き、時間方向の情報伝播を説明できる
- [ ] NumPyで `RNNCell` と `RNN` クラスを **ゼロから実装** できる
- [ ] **勾配チェック** により実装の正しさを検証できる

### 前提知識

- Notebook 70-76（ニューラルネットワークの基礎、逆伝播）
- Notebook 120（シーケンスモデリング入門）

### 難易度・所要時間

- 難易度: ★★★☆☆
- 推定時間: 120-150分

---

## 目次

1. [RNNの状態方程式](#1-rnnの状態方程式)
2. [RNNCellの実装](#2-rnncellの実装)
3. [RNNクラスの実装](#3-rnnクラスの実装)
4. [展開計算グラフの可視化](#4-展開計算グラフの可視化)
5. [フォワードパスのデモ: 正弦波予測](#5-フォワードパスのデモ-正弦波予測)
6. [隠れ状態のヒートマップ](#6-隠れ状態のヒートマップ)
7. [勾配チェック](#7-勾配チェック)
8. [正弦波予測: MLPとの比較](#8-正弦波予測-mlpとの比較)
9. [まとめ・チートシート](#9-まとめチートシート)
10. [自己評価クイズ](#10-自己評価クイズ)
11. [よくある間違い](#11-よくある間違い)

In [ ]:
# ============================================================
# 環境セットアップ
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = ['Hiragino Sans', 'Arial Unicode MS', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

np.random.seed(42)
print("環境セットアップ完了")

---

## 1. RNNの状態方程式

### 1.1 なぜ「状態」が必要か？

フィードフォワードネットワーク（MLP）は、入力に対して **記憶を持たない** 関数です。
しかし、シーケンスデータ（時系列、テキストなど）では、**過去の情報が現在の判断に影響** します。

RNN（Recurrent Neural Network）は、**隠れ状態 $h_t$** を導入することで、
過去の情報を蓄積しながらシーケンスを処理します。

### 1.2 状態方程式

RNNの中核は以下の2つの方程式です：

$$h_t = \tanh(W_{hh} \, h_{t-1} + W_{xh} \, x_t + b_h)$$

$$y_t = W_{hy} \, h_t + b_y$$

| 記号 | 意味 | 形状 |
|------|------|------|
| $x_t$ | 時刻 $t$ の入力 | $(D_{\text{in}},)$ |
| $h_t$ | 時刻 $t$ の隠れ状態 | $(D_h,)$ |
| $y_t$ | 時刻 $t$ の出力 | $(D_{\text{out}},)$ |
| $W_{xh}$ | 入力→隠れ層の重み | $(D_h, D_{\text{in}})$ |
| $W_{hh}$ | 隠れ→隠れ層の重み（再帰） | $(D_h, D_h)$ |
| $W_{hy}$ | 隠れ→出力層の重み | $(D_{\text{out}}, D_h)$ |
| $b_h, b_y$ | バイアス | $(D_h,), (D_{\text{out}},)$ |

### 1.3 重み共有

RNNの最も重要な特徴は **重み共有** です。全ての時刻で **同じ** $W_{xh}, W_{hh}, W_{hy}$ を使います。

これにより：
- **任意の長さ** のシーケンスを処理できる
- パラメータ数がシーケンス長に **依存しない**
- 時刻に関係なく **同じ変換** が適用される（時間的並進不変性）

In [ ]:
# ============================================================
# 1.4 状態方程式の動作を直感的に理解する
# ============================================================

# 簡単な例: 隠れ状態の次元=2, 入力の次元=1
D_in, D_h, D_out = 1, 2, 1

# 手動で重みを設定（理解のため）
W_xh = np.array([[0.5], [0.3]])       # (D_h, D_in)
W_hh = np.array([[0.8, -0.1],
                  [0.2, 0.6]])          # (D_h, D_h)
b_h = np.zeros(D_h)                    # (D_h,)

# 入力シーケンス
x_seq = np.array([1.0, 0.5, -0.3, 0.8, 0.0])  # 長さ5の入力

# 隠れ状態の初期値
h = np.zeros(D_h)

print("=" * 60)
print("RNNの状態方程式: ステップごとの計算")
print("=" * 60)

hidden_states = [h.copy()]

for t, x_t in enumerate(x_seq):
    # 状態方程式: h_t = tanh(W_hh @ h_{t-1} + W_xh @ x_t + b_h)
    x_t_vec = np.array([x_t])  # スカラーをベクトルに
    
    recurrent = W_hh @ h          # 前の隠れ状態からの寄与
    input_contrib = W_xh @ x_t_vec  # 入力からの寄与
    pre_activation = recurrent + input_contrib + b_h
    h = np.tanh(pre_activation)
    
    hidden_states.append(h.copy())
    
    print(f"\nt={t}: x_t = {x_t:.1f}")
    print(f"  再帰項 (W_hh @ h_{{t-1}}): [{recurrent[0]:.4f}, {recurrent[1]:.4f}]")
    print(f"  入力項 (W_xh @ x_t):      [{input_contrib[0]:.4f}, {input_contrib[1]:.4f}]")
    print(f"  h_t = tanh(合計):          [{h[0]:.4f}, {h[1]:.4f}]")

hidden_states = np.array(hidden_states)
print("\n" + "=" * 60)
print("隠れ状態は過去の入力の情報を蓄積しています")

In [ ]:
# ============================================================
# 1.5 隠れ状態の遷移を可視化
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: 隠れ状態の各次元の時間変化
time_steps = np.arange(len(hidden_states))
axes[0].plot(time_steps, hidden_states[:, 0], 'bo-', label='$h_t[0]$', linewidth=2, markersize=8)
axes[0].plot(time_steps, hidden_states[:, 1], 'rs-', label='$h_t[1]$', linewidth=2, markersize=8)

# 入力もプロット（t=0の初期状態は入力なし）
for t, x_t in enumerate(x_seq):
    axes[0].annotate(f'$x_{{{t}}}$={x_t:.1f}', 
                     xy=(t+1, hidden_states[t+1, 0]),
                     xytext=(t+1, hidden_states[t+1, 0] + 0.15),
                     fontsize=9, ha='center', color='gray')

axes[0].set_xlabel('時刻 $t$')
axes[0].set_ylabel('隠れ状態の値')
axes[0].set_title('隠れ状態の時間変化')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(time_steps)
axes[0].set_xticklabels(['初期'] + [f't={t}' for t in range(len(x_seq))])

# 右: 隠れ状態の2D空間での軌跡
axes[1].plot(hidden_states[:, 0], hidden_states[:, 1], 'g-', linewidth=1.5, alpha=0.5)
for i, (h0, h1) in enumerate(hidden_states):
    color = plt.cm.viridis(i / len(hidden_states))
    axes[1].scatter(h0, h1, color=color, s=100, zorder=5, edgecolors='black')
    label = '初期' if i == 0 else f't={i-1}'
    axes[1].annotate(label, (h0, h1), textcoords="offset points", 
                     xytext=(8, 8), fontsize=9)

axes[1].set_xlabel('$h_t[0]$')
axes[1].set_ylabel('$h_t[1]$')
axes[1].set_title('隠れ状態の2D軌跡')
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

print("【観察】")
print("- 隠れ状態は入力に応じて動的に変化する")
print("- 同じ入力でも、過去の履歴によって異なる隠れ状態になる")
print("- tanh により、隠れ状態の値は [-1, 1] の範囲に収まる")

---

## 2. RNNCellの実装

### 2.1 設計方針

`RNNCell` は **1時刻分** の処理を担当するモジュールです。

```
入力: x_t (D_in,), h_prev (D_h,)
出力: h_next (D_h,)

h_next = tanh(W_hh @ h_prev + W_xh @ x_t + b_h)
```

バッチ処理にも対応するため、入力の形状は `(batch_size, D_in)` とします。

In [ ]:
# ============================================================
# 2.2 RNNCellクラスの実装
# ============================================================

class RNNCell:
    """
    Vanilla RNN の1時刻分の処理を行うセル
    
    状態方程式:
        h_t = tanh(x_t @ W_xh.T + h_{t-1} @ W_hh.T + b_h)
    
    Parameters
    ----------
    input_size : int
        入力の次元数 D_in
    hidden_size : int
        隠れ状態の次元数 D_h
    """
    
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # ----- 重みの初期化 (Xavier) -----
        scale_xh = np.sqrt(2.0 / (input_size + hidden_size))
        scale_hh = np.sqrt(2.0 / (hidden_size + hidden_size))
        
        self.W_xh = np.random.randn(hidden_size, input_size) * scale_xh  # (D_h, D_in)
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale_hh # (D_h, D_h)
        self.b_h = np.zeros(hidden_size)                                  # (D_h,)
        
        # ----- 勾配の保存用 -----
        self.dW_xh = np.zeros_like(self.W_xh)
        self.dW_hh = np.zeros_like(self.W_hh)
        self.db_h = np.zeros_like(self.b_h)
        
        # ----- キャッシュ -----
        self.cache = {}
    
    def forward(self, x_t, h_prev):
        """
        1時刻分の順伝播
        
        Parameters
        ----------
        x_t : ndarray, shape (batch_size, input_size)
            時刻 t の入力
        h_prev : ndarray, shape (batch_size, hidden_size)
            時刻 t-1 の隠れ状態
        
        Returns
        -------
        h_next : ndarray, shape (batch_size, hidden_size)
            時刻 t の隠れ状態
        """
        # 状態方程式
        # h_t = tanh(x_t @ W_xh.T + h_{t-1} @ W_hh.T + b_h)
        z = x_t @ self.W_xh.T + h_prev @ self.W_hh.T + self.b_h
        h_next = np.tanh(z)
        
        # 逆伝播のためにキャッシュ
        self.cache = {
            'x_t': x_t,
            'h_prev': h_prev,
            'h_next': h_next,
            'z': z
        }
        
        return h_next
    
    def backward(self, dh_next):
        """
        1時刻分の逆伝播
        
        Parameters
        ----------
        dh_next : ndarray, shape (batch_size, hidden_size)
            h_t に対する勾配
        
        Returns
        -------
        dx_t : ndarray, shape (batch_size, input_size)
            x_t に対する勾配
        dh_prev : ndarray, shape (batch_size, hidden_size)
            h_{t-1} に対する勾配
        """
        x_t = self.cache['x_t']
        h_prev = self.cache['h_prev']
        h_next = self.cache['h_next']
        
        # tanh の微分: dtanh = 1 - tanh^2
        dz = dh_next * (1 - h_next ** 2)  # (batch_size, hidden_size)
        
        # 各パラメータの勾配
        self.dW_xh += dz.T @ x_t          # (D_h, D_in)
        self.dW_hh += dz.T @ h_prev        # (D_h, D_h)
        self.db_h += np.sum(dz, axis=0)    # (D_h,)
        
        # 入力と前の隠れ状態への勾配
        dx_t = dz @ self.W_xh              # (batch_size, D_in)
        dh_prev = dz @ self.W_hh           # (batch_size, D_h)
        
        return dx_t, dh_prev
    
    def reset_grads(self):
        """勾配をゼロにリセット"""
        self.dW_xh = np.zeros_like(self.W_xh)
        self.dW_hh = np.zeros_like(self.W_hh)
        self.db_h = np.zeros_like(self.b_h)


# テスト
print("=" * 60)
print("RNNCell のテスト")
print("=" * 60)

cell = RNNCell(input_size=3, hidden_size=4)
print(f"入力次元: {cell.input_size}")
print(f"隠れ状態次元: {cell.hidden_size}")
print(f"W_xh の形状: {cell.W_xh.shape}")
print(f"W_hh の形状: {cell.W_hh.shape}")
print(f"b_h の形状:  {cell.b_h.shape}")

# 順伝播テスト
batch_size = 2
x_t = np.random.randn(batch_size, 3)
h_prev = np.zeros((batch_size, 4))
h_next = cell.forward(x_t, h_prev)
print(f"\n入力 x_t の形状:    {x_t.shape}")
print(f"出力 h_next の形状: {h_next.shape}")
print(f"h_next の値域: [{h_next.min():.4f}, {h_next.max():.4f}]")
print("\n全ての値が [-1, 1] の範囲内（tanh の出力）")

---

## 3. RNNクラスの実装

### 3.1 設計方針

`RNN` クラスは `RNNCell` を繰り返し呼び出して、**シーケンス全体** を処理します。

```
入力: X = [x_0, x_1, ..., x_{T-1}]  shape: (batch_size, T, D_in)
出力: H = [h_0, h_1, ..., h_{T-1}]  shape: (batch_size, T, D_h)
       Y = [y_0, y_1, ..., y_{T-1}]  shape: (batch_size, T, D_out)
```

In [ ]:
# ============================================================
# 3.2 RNNクラスの実装
# ============================================================

class RNN:
    """
    Vanilla RNN: シーケンス全体を処理するクラス
    
    内部で RNNCell を繰り返し呼び出し、
    全時刻の隠れ状態と出力を計算する。
    
    Parameters
    ----------
    input_size : int
        入力の次元数
    hidden_size : int
        隠れ状態の次元数
    output_size : int
        出力の次元数
    """
    
    def __init__(self, input_size, hidden_size, output_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # RNNセル
        self.cell = RNNCell(input_size, hidden_size)
        
        # 出力層の重み
        scale_hy = np.sqrt(2.0 / (hidden_size + output_size))
        self.W_hy = np.random.randn(output_size, hidden_size) * scale_hy  # (D_out, D_h)
        self.b_y = np.zeros(output_size)                                   # (D_out,)
        
        # 出力層の勾配
        self.dW_hy = np.zeros_like(self.W_hy)
        self.db_y = np.zeros_like(self.b_y)
        
        # 全時刻の隠れ状態を保存
        self.hidden_states = []
        self.outputs = []
        self.T = 0
        self.h0 = None
    
    def forward(self, X, h0=None):
        """
        シーケンス全体の順伝播
        
        Parameters
        ----------
        X : ndarray, shape (batch_size, T, input_size)
            入力シーケンス
        h0 : ndarray, shape (batch_size, hidden_size), optional
            初期隠れ状態（デフォルトはゼロ）
        
        Returns
        -------
        Y : ndarray, shape (batch_size, T, output_size)
            出力シーケンス
        H : ndarray, shape (batch_size, T, hidden_size)
            全時刻の隠れ状態
        """
        batch_size, T, _ = X.shape
        self.T = T
        
        # 初期隠れ状態
        if h0 is None:
            h0 = np.zeros((batch_size, self.hidden_size))
        self.h0 = h0
        
        # 全時刻の隠れ状態と出力を格納
        self.hidden_states = []
        self.outputs = []
        self.cell_caches = []  # 各時刻の RNNCell のキャッシュ
        
        h_prev = h0
        
        for t in range(T):
            x_t = X[:, t, :]  # (batch_size, input_size)
            
            # RNNセルの順伝播
            h_next = self.cell.forward(x_t, h_prev)
            self.cell_caches.append(dict(self.cell.cache))  # キャッシュを保存
            
            # 出力層
            y_t = h_next @ self.W_hy.T + self.b_y  # (batch_size, output_size)
            
            self.hidden_states.append(h_next)
            self.outputs.append(y_t)
            
            h_prev = h_next
        
        # リストを配列に変換
        H = np.stack(self.hidden_states, axis=1)  # (batch_size, T, hidden_size)
        Y = np.stack(self.outputs, axis=1)         # (batch_size, T, output_size)
        
        return Y, H
    
    def backward(self, dY):
        """
        BPTT (Backpropagation Through Time): 時間方向の逆伝播
        
        Parameters
        ----------
        dY : ndarray, shape (batch_size, T, output_size)
            出力に対する勾配
        
        Returns
        -------
        dX : ndarray, shape (batch_size, T, input_size)
            入力に対する勾配
        """
        batch_size = dY.shape[0]
        dX = np.zeros((batch_size, self.T, self.input_size))
        
        # 勾配のリセット
        self.cell.reset_grads()
        self.dW_hy = np.zeros_like(self.W_hy)
        self.db_y = np.zeros_like(self.b_y)
        
        # 時間方向に逆向きに伝播
        dh_next = np.zeros((batch_size, self.hidden_size))
        
        for t in reversed(range(self.T)):
            dy_t = dY[:, t, :]  # (batch_size, output_size)
            
            # 出力層の勾配
            self.dW_hy += dy_t.T @ self.hidden_states[t]  # (D_out, D_h)
            self.db_y += np.sum(dy_t, axis=0)              # (D_out,)
            
            # 隠れ状態への勾配 = 出力層からの勾配 + 次の時刻からの勾配
            dh = dy_t @ self.W_hy + dh_next  # (batch_size, D_h)
            
            # RNNセルの逆伝播
            self.cell.cache = self.cell_caches[t]  # キャッシュを復元
            dx_t, dh_prev = self.cell.backward(dh)
            
            dX[:, t, :] = dx_t
            dh_next = dh_prev
        
        return dX
    
    def reset_grads(self):
        """全勾配をリセット"""
        self.cell.reset_grads()
        self.dW_hy = np.zeros_like(self.W_hy)
        self.db_y = np.zeros_like(self.b_y)
    
    def get_params(self):
        """全パラメータを辞書で返す"""
        return {
            'W_xh': self.cell.W_xh,
            'W_hh': self.cell.W_hh,
            'b_h': self.cell.b_h,
            'W_hy': self.W_hy,
            'b_y': self.b_y
        }
    
    def get_grads(self):
        """全勾配を辞書で返す"""
        return {
            'W_xh': self.cell.dW_xh,
            'W_hh': self.cell.dW_hh,
            'b_h': self.cell.db_h,
            'W_hy': self.dW_hy,
            'b_y': self.db_y
        }


# テスト
print("=" * 60)
print("RNN クラスのテスト")
print("=" * 60)

np.random.seed(42)
rnn = RNN(input_size=3, hidden_size=8, output_size=1)

# ダミー入力
batch_size, T, D_in = 4, 10, 3
X = np.random.randn(batch_size, T, D_in)

# 順伝播
Y, H = rnn.forward(X)

print(f"入力 X の形状:     {X.shape}  (batch={batch_size}, T={T}, D_in={D_in})")
print(f"出力 Y の形状:     {Y.shape}  (batch={batch_size}, T={T}, D_out=1)")
print(f"隠れ状態 H の形状: {H.shape}  (batch={batch_size}, T={T}, D_h=8)")

# パラメータ数
n_params = sum(p.size for p in rnn.get_params().values())
print(f"\n総パラメータ数: {n_params}")
print("（シーケンス長Tに依存しないことに注目）")

---

## 4. 展開計算グラフの可視化

### 4.1 RNNの展開

RNNの再帰構造を時間方向に **展開 (unroll)** すると、
フィードフォワードネットワークのように見えます。
ただし、**全時刻で同じ重みを共有** している点が重要です。

In [ ]:
# ============================================================
# 4.2 展開計算グラフの描画
# ============================================================

fig, ax = plt.subplots(figsize=(14, 7))

T_draw = 5  # 展開する時刻数

# ---- ノードの位置 ----
x_positions = np.arange(T_draw) * 2.5
y_input = 0.0
y_hidden = 1.5
y_output = 3.0

# ---- 色の定義 ----
color_input = '#4ECDC4'
color_hidden = '#FF6B6B'
color_output = '#45B7D1'
color_weight = '#96CEB4'

for t in range(T_draw):
    x = x_positions[t]
    
    # 入力ノード
    circle_in = plt.Circle((x, y_input), 0.35, color=color_input, ec='black', lw=1.5, zorder=5)
    ax.add_patch(circle_in)
    ax.text(x, y_input, f'$x_{{{t}}}$', ha='center', va='center', fontsize=12, fontweight='bold')
    
    # 隠れ状態ノード
    circle_h = plt.Circle((x, y_hidden), 0.4, color=color_hidden, ec='black', lw=1.5, zorder=5)
    ax.add_patch(circle_h)
    ax.text(x, y_hidden, f'$h_{{{t}}}$', ha='center', va='center', fontsize=12, fontweight='bold')
    
    # 出力ノード
    circle_out = plt.Circle((x, y_output), 0.35, color=color_output, ec='black', lw=1.5, zorder=5)
    ax.add_patch(circle_out)
    ax.text(x, y_output, f'$y_{{{t}}}$', ha='center', va='center', fontsize=12, fontweight='bold')
    
    # 入力 → 隠れ の矢印
    ax.annotate('', xy=(x, y_hidden - 0.4), xytext=(x, y_input + 0.35),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.5))
    if t == 0:
        ax.text(x + 0.25, (y_input + y_hidden) / 2, '$W_{xh}$', fontsize=10, color='#2C3E50')
    
    # 隠れ → 出力 の矢印
    ax.annotate('', xy=(x, y_output - 0.35), xytext=(x, y_hidden + 0.4),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.5))
    if t == 0:
        ax.text(x + 0.25, (y_hidden + y_output) / 2, '$W_{hy}$', fontsize=10, color='#2C3E50')
    
    # 隠れ → 隠れ の矢印（再帰結合）
    if t < T_draw - 1:
        x_next = x_positions[t + 1]
        ax.annotate('', xy=(x_next - 0.4, y_hidden), xytext=(x + 0.4, y_hidden),
                    arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2.5))
        if t == 0:
            ax.text((x + x_next) / 2, y_hidden + 0.3, '$W_{hh}$', 
                    fontsize=10, color='#E74C3C', ha='center', fontweight='bold')

# 初期状態
ax.annotate('', xy=(x_positions[0] - 0.4, y_hidden), xytext=(x_positions[0] - 1.2, y_hidden),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, linestyle='--'))
ax.text(x_positions[0] - 1.6, y_hidden, '$h_0$', fontsize=12, ha='center', va='center', color='gray')

# 凡例的な説明
ax.text(x_positions[-1] + 1.5, y_output, '同じ重み\n$W_{xh}, W_{hh}, W_{hy}$\nを全時刻で共有',
        fontsize=10, ha='left', va='center',
        bbox=dict(boxstyle='round,pad=0.5', facecolor=color_weight, alpha=0.3))

ax.set_xlim(-2.5, x_positions[-1] + 4.5)
ax.set_ylim(-0.8, 3.8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('RNNの展開計算グラフ (Unrolled Computation Graph)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("【ポイント】")
print("- 赤い矢印 (W_hh) が再帰結合: 隠れ状態が次の時刻に伝播される")
print("- 全時刻で W_xh, W_hh, W_hy は同一の重み（重み共有）")
print("- 展開すると深いネットワークに見える → 勾配消失・爆発の原因")

---

## 5. フォワードパスのデモ: 正弦波予測

### 5.1 正弦波データの準備

RNNの動作を確認するために、正弦波データを使った予測タスクを行います。
ここでは学習（パラメータ更新）は行わず、**フォワードパスのみ** を確認します。

In [ ]:
# ============================================================
# 5.2 正弦波データの生成
# ============================================================

def generate_sine_data(n_samples=100, seq_len=25, freq=0.1, noise=0.0):
    """
    正弦波データを生成
    
    タスク: x_0, ..., x_{T-1} から y_0, ..., y_{T-1} を予測
    ここで y_t = sin(次の時刻) とする（1ステップ先の予測）
    """
    X = []
    Y = []
    
    for i in range(n_samples):
        # ランダムな位相オフセット
        phase = np.random.uniform(0, 2 * np.pi)
        
        # 時系列の生成
        t = np.arange(seq_len + 1) * freq * 2 * np.pi + phase
        signal = np.sin(t)
        
        # 入力: x_0, ..., x_{T-1}
        x = signal[:-1].reshape(-1, 1)  # (seq_len, 1)
        # ターゲット: y = x_{1}, ..., x_{T}
        y = signal[1:].reshape(-1, 1)   # (seq_len, 1)
        
        if noise > 0:
            x += np.random.randn(*x.shape) * noise
        
        X.append(x)
        Y.append(y)
    
    return np.array(X), np.array(Y)


np.random.seed(42)
X_train, Y_train = generate_sine_data(n_samples=200, seq_len=25)
X_test, Y_test = generate_sine_data(n_samples=50, seq_len=25)

print(f"訓練データ: X={X_train.shape}, Y={Y_train.shape}")
print(f"テストデータ: X={X_test.shape}, Y={Y_test.shape}")

# データの可視化
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, ax in enumerate(axes.flat):
    ax.plot(X_train[i, :, 0], 'b.-', label='入力 $x_t$', linewidth=1.5)
    ax.plot(Y_train[i, :, 0], 'r.--', label='ターゲット $y_t$', linewidth=1.5)
    ax.set_xlabel('時刻 $t$')
    ax.set_ylabel('値')
    ax.set_title(f'サンプル {i+1}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('正弦波データ: 1ステップ先予測タスク', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5.3 フォワードパスの実行（学習前）
# ============================================================

np.random.seed(42)
rnn_sine = RNN(input_size=1, hidden_size=16, output_size=1)

# ランダムな重みでの予測（学習前）
Y_pred_before, H_before = rnn_sine.forward(X_test[:5])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, ax in enumerate(axes):
    ax.plot(X_test[i, :, 0], 'b.-', label='入力', linewidth=1.5)
    ax.plot(Y_test[i, :, 0], 'r.--', label='ターゲット', linewidth=1.5)
    ax.plot(Y_pred_before[i, :, 0], 'g.-', label='予測（学習前）', linewidth=1.5)
    ax.set_xlabel('時刻 $t$')
    ax.set_title(f'サンプル {i+1}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('学習前のRNN予測（ランダムな重み）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 学習前の損失
loss_before = np.mean((Y_pred_before - Y_test[:5]) ** 2)
print(f"学習前の MSE Loss: {loss_before:.4f}")
print("ランダムな重みなので、当然予測はデタラメ")

In [ ]:
# ============================================================
# 5.4 簡易学習ループ
# ============================================================

np.random.seed(42)
rnn_train = RNN(input_size=1, hidden_size=16, output_size=1)

# ハイパーパラメータ
learning_rate = 0.005
n_epochs = 200
batch_size = 32
clip_value = 5.0  # 勾配クリッピング

losses = []

for epoch in range(n_epochs):
    # ミニバッチ
    idx = np.random.choice(len(X_train), batch_size, replace=False)
    X_batch = X_train[idx]
    Y_batch = Y_train[idx]
    
    # 順伝播
    Y_pred, H = rnn_train.forward(X_batch)
    
    # 損失 (MSE)
    loss = np.mean((Y_pred - Y_batch) ** 2)
    losses.append(loss)
    
    # 逆伝播
    dY = 2.0 * (Y_pred - Y_batch) / (batch_size * Y_batch.shape[1] * Y_batch.shape[2])
    rnn_train.backward(dY)
    
    # 勾配クリッピング
    grads = rnn_train.get_grads()
    total_norm = np.sqrt(sum(np.sum(g**2) for g in grads.values()))
    if total_norm > clip_value:
        scale = clip_value / total_norm
        for key in grads:
            grads[key] *= scale
    
    # パラメータ更新 (SGD)
    params = rnn_train.get_params()
    params['W_xh'] -= learning_rate * grads['W_xh']
    params['W_hh'] -= learning_rate * grads['W_hh']
    params['b_h'] -= learning_rate * grads['b_h']
    params['W_hy'] -= learning_rate * grads['W_hy']
    params['b_y'] -= learning_rate * grads['b_y']
    
    # パラメータを反映
    rnn_train.cell.W_xh = params['W_xh']
    rnn_train.cell.W_hh = params['W_hh']
    rnn_train.cell.b_h = params['b_h']
    rnn_train.W_hy = params['W_hy']
    rnn_train.b_y = params['b_y']
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {loss:.6f} | Grad Norm: {total_norm:.4f}")

# 学習曲線
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses, linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('RNNの学習曲線（正弦波予測）')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5.5 学習後の予測結果
# ============================================================

Y_pred_after, H_after = rnn_train.forward(X_test[:6])

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, ax in enumerate(axes.flat):
    ax.plot(Y_test[i, :, 0], 'r.-', label='ターゲット', linewidth=2)
    ax.plot(Y_pred_after[i, :, 0], 'g.--', label='RNN予測', linewidth=2)
    ax.set_xlabel('時刻 $t$')
    ax.set_ylabel('値')
    ax.set_title(f'テストサンプル {i+1}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('学習後のRNN予測（正弦波）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

loss_after = np.mean((Y_pred_after - Y_test[:6]) ** 2)
print(f"学習前の MSE: {loss_before:.4f}")
print(f"学習後の MSE: {loss_after:.4f}")
print(f"改善率: {(1 - loss_after / loss_before) * 100:.1f}%")

---

## 6. 隠れ状態のヒートマップ

### 6.1 隠れ状態の時間発展

RNNの「記憶」がどのように時間とともに変化するのかを、**ヒートマップ** で可視化します。
各行が隠れ状態の1つの次元、各列が時刻に対応します。

In [ ]:
# ============================================================
# 6.2 隠れ状態ヒートマップの可視化
# ============================================================

# 学習済みRNNで1サンプルの隠れ状態を取得
sample_idx = 0
X_sample = X_test[sample_idx:sample_idx+1]  # (1, T, 1)
Y_pred_sample, H_sample = rnn_train.forward(X_sample)

# H_sample: (1, T, D_h) → (T, D_h) に変換
H_map = H_sample[0]  # (T, D_h)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), 
                          gridspec_kw={'height_ratios': [1, 3, 1]})

# 上段: 入力信号
axes[0].plot(X_sample[0, :, 0], 'b.-', linewidth=2, markersize=6)
axes[0].set_ylabel('入力 $x_t$')
axes[0].set_title('入力信号と隠れ状態の時間発展', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(-0.5, H_map.shape[0] - 0.5)

# 中段: 隠れ状態のヒートマップ
im = axes[1].imshow(H_map.T, aspect='auto', cmap='RdBu_r', 
                     vmin=-1, vmax=1,
                     extent=[-0.5, H_map.shape[0]-0.5, H_map.shape[1]-0.5, -0.5])
axes[1].set_xlabel('時刻 $t$')
axes[1].set_ylabel('隠れユニット番号')
cbar = plt.colorbar(im, ax=axes[1], label='$h_t$ の値')

# 下段: 予測出力
axes[2].plot(Y_test[sample_idx, :, 0], 'r.-', label='ターゲット', linewidth=2, markersize=6)
axes[2].plot(Y_pred_sample[0, :, 0], 'g.--', label='予測', linewidth=2, markersize=6)
axes[2].set_ylabel('出力 $y_t$')
axes[2].set_xlabel('時刻 $t$')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim(-0.5, H_map.shape[0] - 0.5)

plt.tight_layout()
plt.show()

print("【ヒートマップの読み方】")
print("- 赤 (+1に近い): そのユニットが強く活性化")
print("- 青 (-1に近い): そのユニットが強く抑制")
print("- 白 (0に近い): ほぼ不活性")
print("- 各ユニットが入力信号の異なる特徴を捉えていることが分かる")

In [ ]:
# ============================================================
# 6.3 複数サンプルの隠れ状態の比較
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for i in range(6):
    row, col = i // 3, i % 3
    ax = axes[row, col]
    
    X_s = X_test[i:i+1]
    _, H_s = rnn_train.forward(X_s)
    H_s = H_s[0]  # (T, D_h)
    
    im = ax.imshow(H_s.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title(f'サンプル {i+1}', fontsize=11)
    ax.set_xlabel('時刻 $t$')
    if col == 0:
        ax.set_ylabel('隠れユニット')

plt.suptitle('異なる入力に対する隠れ状態の変化パターン', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("【観察】")
print("- 入力の位相が異なると、隠れ状態のパターンも変化する")
print("- しかし、正弦波という共通構造を反映した類似パターンも見られる")

---

## 7. 勾配チェック

### 7.1 数値勾配と解析勾配の比較

実装の正しさを検証する最も確実な方法は、**勾配チェック** です。

**数値勾配（有限差分法）:**

$$\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta_i + \epsilon) - L(\theta_i - \epsilon)}{2\epsilon}$$

ここで $\epsilon = 10^{-5}$ を使用します。

**相対誤差:**

$$\text{relative error} = \frac{|g_{\text{numerical}} - g_{\text{analytical}}|}{\max(|g_{\text{numerical}}|, |g_{\text{analytical}}|) + \epsilon}$$

相対誤差が $10^{-5}$ 未満であれば、実装は正しいと判断できます。

In [ ]:
# ============================================================
# 7.2 勾配チェックの実装
# ============================================================

def compute_loss(rnn, X, Y_target):
    """RNNの損失を計算"""
    Y_pred, _ = rnn.forward(X)
    return np.mean((Y_pred - Y_target) ** 2)


def numerical_gradient(rnn, X, Y_target, param_name, epsilon=1e-5):
    """
    有限差分法による数値勾配の計算
    
    Parameters
    ----------
    rnn : RNN
        RNNモデル
    X : ndarray
        入力データ
    Y_target : ndarray
        ターゲットデータ
    param_name : str
        勾配を計算するパラメータ名
    epsilon : float
        微小変化量
    
    Returns
    -------
    num_grad : ndarray
        数値勾配
    """
    params = rnn.get_params()
    param = params[param_name]
    num_grad = np.zeros_like(param)
    
    # パラメータの各要素について有限差分
    it = np.nditer(param, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        original_val = param[idx]
        
        # f(theta + epsilon)
        param[idx] = original_val + epsilon
        loss_plus = compute_loss(rnn, X, Y_target)
        
        # f(theta - epsilon)
        param[idx] = original_val - epsilon
        loss_minus = compute_loss(rnn, X, Y_target)
        
        # 数値勾配
        num_grad[idx] = (loss_plus - loss_minus) / (2 * epsilon)
        
        # 値を戻す
        param[idx] = original_val
        it.iternext()
    
    return num_grad


# 小さなRNNで勾配チェック
np.random.seed(42)
rnn_check = RNN(input_size=2, hidden_size=3, output_size=1)

# 小さなデータ
X_check = np.random.randn(2, 4, 2)  # batch=2, T=4, D_in=2
Y_check = np.random.randn(2, 4, 1)  # batch=2, T=4, D_out=1

# 解析勾配の計算
Y_pred_check, _ = rnn_check.forward(X_check)
dY_check = 2.0 * (Y_pred_check - Y_check) / (X_check.shape[0] * X_check.shape[1] * Y_check.shape[2])
rnn_check.backward(dY_check)
analytical_grads = rnn_check.get_grads()

print("=" * 60)
print("勾配チェック: 数値勾配 vs 解析勾配")
print("=" * 60)

results = {}

for param_name in ['W_xh', 'W_hh', 'b_h', 'W_hy', 'b_y']:
    num_grad = numerical_gradient(rnn_check, X_check, Y_check, param_name)
    ana_grad = analytical_grads[param_name]
    
    # 相対誤差
    diff = np.abs(num_grad - ana_grad)
    denom = np.maximum(np.abs(num_grad), np.abs(ana_grad)) + 1e-8
    rel_error = np.max(diff / denom)
    
    results[param_name] = rel_error
    
    status = "OK" if rel_error < 1e-5 else "NG"
    print(f"  {param_name:6s} | 相対誤差: {rel_error:.2e} | {status}")

print("\n" + "-" * 40)
max_error = max(results.values())
if max_error < 1e-5:
    print(f"全パラメータの勾配チェック: PASSED (最大誤差: {max_error:.2e})")
else:
    print(f"勾配チェック: FAILED (最大誤差: {max_error:.2e})")

In [ ]:
# ============================================================
# 7.3 勾配チェック結果の可視化
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, param_name in enumerate(['W_xh', 'W_hh', 'b_h', 'W_hy', 'b_y']):
    row, col = i // 3, i % 3
    ax = axes[row, col]
    
    num_grad = numerical_gradient(rnn_check, X_check, Y_check, param_name)
    ana_grad = analytical_grads[param_name]
    
    # フラットにして散布図
    num_flat = num_grad.flatten()
    ana_flat = ana_grad.flatten()
    
    ax.scatter(num_flat, ana_flat, alpha=0.7, s=50, edgecolors='black', linewidths=0.5)
    
    # 理想線 y=x
    lim_min = min(num_flat.min(), ana_flat.min()) * 1.1
    lim_max = max(num_flat.max(), ana_flat.max()) * 1.1
    ax.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', linewidth=1.5, label='$y=x$')
    
    ax.set_xlabel('数値勾配')
    ax.set_ylabel('解析勾配')
    ax.set_title(f'{param_name} (誤差: {results[param_name]:.1e})')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

# 最後のサブプロットは非表示
axes[1, 2].axis('off')
axes[1, 2].text(0.5, 0.5, '勾配チェック\n全パラメータ\nPASSED',
                ha='center', va='center', fontsize=16, fontweight='bold',
                color='green', transform=axes[1, 2].transAxes,
                bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgreen', alpha=0.3))

plt.suptitle('勾配チェック: 数値勾配 vs 解析勾配', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("【解釈】")
print("- 点が y=x 線上に乗っていれば、解析勾配が正しい")
print("- 全パラメータで相対誤差が 1e-5 未満 → 実装は正しい")

---

## 8. 正弦波予測: MLPとの比較

### 8.1 MLPによる正弦波予測

NB 120 で扱った MLP（Multi-Layer Perceptron）と RNN の予測性能を比較します。

**MLP のアプローチ:** 固定長の過去ウィンドウを入力として使用

**RNN のアプローチ:** 隠れ状態で過去の情報を蓄積

In [ ]:
# ============================================================
# 8.2 簡易MLPの実装と学習
# ============================================================

class SimpleMLP:
    """
    2層MLP: 正弦波予測用
    
    入力: 過去 window_size 個の値
    出力: 次の1つの値
    """
    
    def __init__(self, window_size, hidden_size):
        self.window_size = window_size
        scale1 = np.sqrt(2.0 / (window_size + hidden_size))
        scale2 = np.sqrt(2.0 / (hidden_size + 1))
        
        self.W1 = np.random.randn(hidden_size, window_size) * scale1
        self.b1 = np.zeros(hidden_size)
        self.W2 = np.random.randn(1, hidden_size) * scale2
        self.b2 = np.zeros(1)
        
        self.cache = {}
    
    def forward(self, x):
        """x: (batch_size, window_size)"""
        z1 = x @ self.W1.T + self.b1
        a1 = np.tanh(z1)
        z2 = a1 @ self.W2.T + self.b2
        
        self.cache = {'x': x, 'z1': z1, 'a1': a1}
        return z2
    
    def backward(self, dy):
        """dy: (batch_size, 1)"""
        x, a1 = self.cache['x'], self.cache['a1']
        
        self.dW2 = dy.T @ a1
        self.db2 = np.sum(dy, axis=0)
        
        da1 = dy @ self.W2
        dz1 = da1 * (1 - a1 ** 2)
        
        self.dW1 = dz1.T @ x
        self.db1 = np.sum(dz1, axis=0)


# MLP用のデータ準備（ウィンドウ方式）
window_size = 5

def prepare_mlp_data(X_seq, window_size):
    """シーケンスデータをウィンドウ形式に変換"""
    X_windows = []
    Y_targets = []
    
    for sample in X_seq:
        signal = sample[:, 0]  # (T,)
        for t in range(window_size, len(signal)):
            X_windows.append(signal[t-window_size:t])
            Y_targets.append(signal[t])
    
    return np.array(X_windows), np.array(Y_targets).reshape(-1, 1)


# フルシーケンス（入力+ターゲットを結合）
full_seq_train = np.concatenate([X_train, Y_train[:, -1:, :]], axis=1)  # 最後のターゲットを追加
X_mlp_train, Y_mlp_train = prepare_mlp_data(full_seq_train, window_size)

full_seq_test = np.concatenate([X_test, Y_test[:, -1:, :]], axis=1)
X_mlp_test, Y_mlp_test = prepare_mlp_data(full_seq_test, window_size)

print(f"MLP訓練データ: X={X_mlp_train.shape}, Y={Y_mlp_train.shape}")
print(f"MLPテストデータ: X={X_mlp_test.shape}, Y={Y_mlp_test.shape}")

# MLPの学習
np.random.seed(42)
mlp = SimpleMLP(window_size=window_size, hidden_size=32)

lr_mlp = 0.01
n_epochs_mlp = 300
batch_size_mlp = 64
losses_mlp = []

for epoch in range(n_epochs_mlp):
    idx = np.random.choice(len(X_mlp_train), batch_size_mlp, replace=False)
    X_b = X_mlp_train[idx]
    Y_b = Y_mlp_train[idx]
    
    Y_pred = mlp.forward(X_b)
    loss = np.mean((Y_pred - Y_b) ** 2)
    losses_mlp.append(loss)
    
    dy = 2.0 * (Y_pred - Y_b) / batch_size_mlp
    mlp.backward(dy)
    
    mlp.W1 -= lr_mlp * mlp.dW1
    mlp.b1 -= lr_mlp * mlp.db1
    mlp.W2 -= lr_mlp * mlp.dW2
    mlp.b2 -= lr_mlp * mlp.db2

print(f"\nMLP最終損失: {losses_mlp[-1]:.6f}")

In [ ]:
# ============================================================
# 8.3 RNN vs MLP の比較
# ============================================================

# テストデータでの評価
Y_mlp_pred = mlp.forward(X_mlp_test)
mlp_test_loss = np.mean((Y_mlp_pred - Y_mlp_test) ** 2)

Y_rnn_pred, _ = rnn_train.forward(X_test)
rnn_test_loss = np.mean((Y_rnn_pred - Y_test) ** 2)

print(f"テスト損失の比較:")
print(f"  MLP (window={window_size}): {mlp_test_loss:.6f}")
print(f"  RNN (hidden=16):           {rnn_test_loss:.6f}")

# 可視化: 同じテストサンプルの予測を比較
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, ax in enumerate(axes.flat):
    # RNNの予測
    rnn_pred_i = Y_rnn_pred[i, :, 0]
    target_i = Y_test[i, :, 0]
    
    # MLPの予測（ウィンドウ方式なのでwindow_size分ずれる）
    signal_i = X_test[i, :, 0]
    mlp_pred_list = []
    for t in range(window_size, len(signal_i)):
        window = signal_i[t-window_size:t].reshape(1, -1)
        pred = mlp.forward(window)
        mlp_pred_list.append(pred[0, 0])
    
    ax.plot(target_i, 'k.-', label='ターゲット', linewidth=2, markersize=5)
    ax.plot(rnn_pred_i, 'g.--', label='RNN', linewidth=2, markersize=5)
    t_mlp = np.arange(window_size, len(signal_i))
    ax.plot(t_mlp, mlp_pred_list, 'r.:', label=f'MLP (win={window_size})', linewidth=2, markersize=5)
    
    ax.set_xlabel('時刻 $t$')
    ax.set_ylabel('値')
    ax.set_title(f'テストサンプル {i+1}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('RNN vs MLP: 正弦波予測の比較', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n【RNN vs MLP の違い】")
print("- MLP: 固定ウィンドウ（過去5ステップ）しか見えない")
print("- RNN: 隠れ状態を通じて任意の過去の情報を保持できる（理論上）")
print("- 正弦波のような単純なパターンでは両者とも良い性能")
print("- より長期の依存関係がある場合、RNNが有利")

In [ ]:
# ============================================================
# 8.4 パラメータ効率の比較
# ============================================================

# パラメータ数の比較
rnn_params = sum(p.size for p in rnn_train.get_params().values())
mlp_params = mlp.W1.size + mlp.b1.size + mlp.W2.size + mlp.b2.size

print("=" * 50)
print("パラメータ数の比較")
print("=" * 50)
print(f"\nRNN (input=1, hidden=16, output=1):")
print(f"  W_xh: {rnn_train.cell.W_xh.shape} = {rnn_train.cell.W_xh.size}")
print(f"  W_hh: {rnn_train.cell.W_hh.shape} = {rnn_train.cell.W_hh.size}")
print(f"  b_h:  {rnn_train.cell.b_h.shape}   = {rnn_train.cell.b_h.size}")
print(f"  W_hy: {rnn_train.W_hy.shape}  = {rnn_train.W_hy.size}")
print(f"  b_y:  {rnn_train.b_y.shape}      = {rnn_train.b_y.size}")
print(f"  合計: {rnn_params}")

print(f"\nMLP (window={window_size}, hidden=32, output=1):")
print(f"  W1: {mlp.W1.shape} = {mlp.W1.size}")
print(f"  b1: {mlp.b1.shape}  = {mlp.b1.size}")
print(f"  W2: {mlp.W2.shape}  = {mlp.W2.size}")
print(f"  b2: {mlp.b2.shape}      = {mlp.b2.size}")
print(f"  合計: {mlp_params}")

print(f"\n重要: RNNのパラメータ数はシーケンス長に依存しない")
print(f"  MLPはウィンドウサイズを大きくするとパラメータが増える")

---

## 9. まとめ・チートシート

### 9.1 本ノートブックで学んだこと

| トピック | 要点 |
|---------|------|
| **状態方程式** | $h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$, $y_t = W_{hy} h_t + b_y$ |
| **重み共有** | 全時刻で同じ重みを使用 → パラメータ数がTに依存しない |
| **展開計算グラフ** | 再帰を展開すると深いFFNに見える |
| **RNNCell** | 1時刻分の forward/backward を実装 |
| **RNN** | RNNCellを繰り返し呼び出してシーケンス全体を処理 |
| **BPTT** | 時間方向の逆伝播（Backpropagation Through Time） |
| **勾配チェック** | 数値勾配と解析勾配の比較で実装を検証 |
| **MLP比較** | RNNは可変長入力に対応、パラメータ効率が良い |

### 9.2 チートシート

```python
# --- RNNCell の核心コード ---
# 順伝播（1時刻）
z = x_t @ W_xh.T + h_prev @ W_hh.T + b_h
h_next = np.tanh(z)

# 逆伝播（1時刻）
dz = dh_next * (1 - h_next ** 2)   # tanh の微分
dW_xh += dz.T @ x_t
dW_hh += dz.T @ h_prev
db_h  += dz.sum(axis=0)
dx_t   = dz @ W_xh
dh_prev = dz @ W_hh

# --- RNN のシーケンス処理 ---
for t in range(T):
    h = cell.forward(X[:, t, :], h)

# --- BPTT ---
for t in reversed(range(T)):
    dx_t, dh = cell.backward(dh_total)
```

### 9.3 形状のまとめ

| テンソル | 形状 | 説明 |
|---------|------|------|
| `X` | `(B, T, D_in)` | 入力シーケンス |
| `H` | `(B, T, D_h)` | 隠れ状態の系列 |
| `Y` | `(B, T, D_out)` | 出力シーケンス |
| `W_xh` | `(D_h, D_in)` | 入力→隠れの重み |
| `W_hh` | `(D_h, D_h)` | 隠れ→隠れの重み |
| `W_hy` | `(D_out, D_h)` | 隠れ→出力の重み |

---

## 10. 自己評価クイズ

以下の問いに答えて、理解度を確認しましょう。

---

**Q1.** RNNの状態方程式 $h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$ において、
$W_{hh}$ の役割は何ですか？ なぜ「再帰」結合と呼ばれるのですか？

<details>
<summary>解答を表示</summary>

$W_{hh}$ は **前の時刻の隠れ状態** $h_{t-1}$ を現在の隠れ状態 $h_t$ に変換する重みです。
出力（隠れ状態）が再び入力として使われるため「再帰（recurrent）」と呼ばれます。
この再帰結合により、RNNは過去の情報を蓄積・伝播することができます。
</details>

---

**Q2.** RNNが「重み共有」を行うことの利点を2つ挙げてください。

<details>
<summary>解答を表示</summary>

1. **任意の長さのシーケンスを処理可能**: 同じ重みを繰り返し使うため、シーケンス長Tに制限がない
2. **パラメータ効率が良い**: パラメータ数がシーケンス長Tに依存しない（MLPはウィンドウサイズに比例してパラメータが増える）

補足: 時間的並進不変性（同じ変換が全時刻に適用される）も重要な利点です。
</details>

---

**Q3.** 勾配チェックで使う有限差分法の式を書き、$\epsilon$ の典型的な値を答えてください。

<details>
<summary>解答を表示</summary>

$$\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta_i + \epsilon) - L(\theta_i - \epsilon)}{2\epsilon}$$

典型的な値: $\epsilon = 10^{-5}$

中心差分（2点を使う）は片側差分より精度が高く、$O(\epsilon^2)$ の誤差です。
</details>

---

**Q4.** RNNの展開計算グラフを見たとき、なぜ勾配消失・爆発が起こりやすいと言えますか？

<details>
<summary>解答を表示</summary>

展開するとT層の深いネットワークに等しくなり、逆伝播時に $W_{hh}$ と $\tanh'$ の積が
T回繰り返されます。

- $\|W_{hh}\| < 1$ かつ $\tanh' < 1$ の場合: 勾配が指数的に減衰（勾配消失）
- $\|W_{hh}\| > 1$ の場合: 勾配が指数的に増大（勾配爆発）

これがVanilla RNNの根本的な弱点であり、LSTMやGRUが開発された動機です。
</details>

---

**Q5.** `RNNCell.backward()` で `dW_hh += dz.T @ h_prev` としているのに、
`dW_hh = dz.T @ h_prev` としないのはなぜですか？

<details>
<summary>解答を表示</summary>

**重み共有** のためです。全時刻で同じ $W_{hh}$ を使っているため、
各時刻での勾配を **加算（累積）** する必要があります。

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=0}^{T-1} \frac{\partial L_t}{\partial W_{hh}}$$

`+=` により、BPTT で各時刻の寄与を正しく累積しています。
</details>

---

## 11. よくある間違い

### 間違い1: 勾配のリセット忘れ

```python
# NG: 勾配が前のバッチの値に累積される
for epoch in range(n_epochs):
    rnn.forward(X)
    rnn.backward(dY)
    # 勾配リセットなし!

# OK: 毎回リセット
for epoch in range(n_epochs):
    rnn.reset_grads()        # <- ここが重要
    rnn.forward(X)
    rnn.backward(dY)
```

### 間違い2: 隠れ状態の初期化

```python
# NG: 前のバッチの隠れ状態が残る
h = rnn.hidden_states[-1]  # 前のバッチの最終状態
Y, H = rnn.forward(X_new, h0=h)

# OK: 通常はゼロで初期化（バッチ間で独立）
Y, H = rnn.forward(X_new, h0=None)  # 内部でゼロ初期化
```

### 間違い3: 勾配クリッピングなしの学習

```python
# NG: 勾配爆発でパラメータが発散
params['W_hh'] -= lr * grads['W_hh']  # 勾配が巨大だと破綻

# OK: 勾配ノルムをクリッピング
total_norm = np.sqrt(sum(np.sum(g**2) for g in grads.values()))
if total_norm > max_norm:
    for key in grads:
        grads[key] *= max_norm / total_norm
```

### 間違い4: BPTT での勾配の伝播方向

```python
# NG: 時間方向を正順に逆伝播
for t in range(T):                  # 間違い!
    dx_t, dh = cell.backward(dh)

# OK: 時間方向を逆順に逆伝播
for t in reversed(range(T)):        # 正しい
    dx_t, dh = cell.backward(dh)
```

### 間違い5: tanh の微分の間違い

```python
# NG: sigmoid の微分と混同
dtanh = h_next * (1 - h_next)       # これは sigmoid' !

# OK: tanh の微分
dtanh = 1 - h_next ** 2             # tanh'(z) = 1 - tanh(z)^2
```

---

## 参考文献

1. Elman, J. L. (1990). Finding structure in time. *Cognitive Science*, 14(2), 179-211.
2. Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature*, 323, 533-536.
3. Pascanu, R., Mikolov, T., & Bengio, Y. (2013). On the difficulty of training recurrent neural networks. *ICML*.
4. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*, Chapter 10: Sequence Modeling.

---

**次のノートブック:** LSTM/GRU ― 勾配消失を克服するゲート機構